# generator.ipynb

# Assignment 4
### CSC 537: Deep Learning

**Author:** Xander Palermo — ajp2s@missouristate.edu <br>
**Professor:** Mukulika Ghosh <br>
**Date:** 27 April 2026

-----

### Imports

Libraries

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
import requests
import pickle


Custom dataset class

In [2]:
from ShakespeareDataset import ShakespeareDataset

Global Variables

In [3]:
SEQUENCE_LENGTH = 50
SHORT_SEQUENCE_LENGTH = 25
RAW_TEXT_PATH = "dataset/tiny_shakespeare/tiny_shakespeare.txt"
DUMP_PATH = "dataset/"
TRANSLATOR_PATH = 'translator/'

BATCH_SIZE = 256

## Generate Data

#### Obtain dataset

In [4]:
raw_data = None

if not os.path.exists(RAW_TEXT_PATH):      # Request raw dataset and save to disk
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    raw_data = response.text

    with open(RAW_TEXT_PATH, "w") as f:         # Cache file
        f.write(raw_data)

else:                                       # Cache raw dataset file
    with open(RAW_TEXT_PATH, "r") as f:
        raw_data = f.read()

raw_data = str(raw_data)

#### Preprocess

In [5]:
raw_data = raw_data.lower()

In [6]:
char = sorted(set(raw_data))

char2idx = {ch: i for i, ch in enumerate(char)}
idx2char = {i : ch for i, ch in enumerate(char)}

In [7]:
raw_data = [char2idx[char] for char in raw_data]
raw_data = torch.tensor(raw_data, dtype=torch.long)

#### Create DataLoader & Save

This dataloader is used as a constant for all experiments

*Training Set*

In [8]:
training_dataset = ShakespeareDataset(raw_data, SEQUENCE_LENGTH)
training_dataloader = DataLoader(training_dataset, batch_size=BATCH_SIZE, shuffle=True)

*Validation Set*

In [9]:
validation_dataset = ShakespeareDataset(raw_data, SEQUENCE_LENGTH)
validation_dataloader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=True)

*Testing Set*

In [10]:
testing_dataset = ShakespeareDataset(raw_data, SEQUENCE_LENGTH)
testing_dataloader = DataLoader(testing_dataset, batch_size=BATCH_SIZE, shuffle=True)

This dataloader is used as an independent variable to test the impact of sequence length.

In [11]:
short_seq_dataset = ShakespeareDataset(raw_data, SHORT_SEQUENCE_LENGTH)
short_dataloader = DataLoader(short_seq_dataset, batch_size=BATCH_SIZE, shuffle=True)

short_seq_validation_dataset = ShakespeareDataset(raw_data, SHORT_SEQUENCE_LENGTH)
short_validation_dataloader = DataLoader(short_seq_validation_dataset, batch_size=BATCH_SIZE, shuffle=True)

short_seq_testing_dataset = ShakespeareDataset(raw_data, SHORT_SEQUENCE_LENGTH)
short_testing_dataloader = DataLoader(short_seq_testing_dataset, batch_size=BATCH_SIZE, shuffle=True)

Save all dataloaders to disk

In [12]:
with open(DUMP_PATH + "training_norm_shakespeare.pkl", "wb") as f:
    pickle.dump(training_dataloader, f)

with open(DUMP_PATH + "validation_shakespeare.pkl", "wb") as f:
    pickle.dump(validation_dataloader, f)

with open(DUMP_PATH + "testing_shakespeare.pkl", "wb") as f:
    pickle.dump(testing_dataloader, f)

with open(DUMP_PATH + "training_short_shakespeare.pkl", "wb") as f:
    pickle.dump(short_dataloader, f)

with open(DUMP_PATH + "validation_short_shakespeare.pkl", "wb") as f:
    pickle.dump(short_validation_dataloader, f)

with open(DUMP_PATH + "testing_short_shakespeare.pkl", "wb") as f:
    pickle.dump(short_testing_dataloader, f)

In [13]:
package = [char2idx, idx2char]

with open(DUMP_PATH + TRANSLATOR_PATH + "translator.pkl", "wb") as f:
    pickle.dump(package, f)

#### Validate Shapes

Turn token ids into readable characters

In [14]:
def translate_to_string(tokens):
    return ''.join([idx2char[idx] for idx in tokens]).replace('\n','\\').replace(' ','_')

In [15]:
print("Normal Dataset:")


input, target = training_dataset[0]

print(f"{' ':11}{'Length':^20}{' ':10}{'Raw':^50}")
print("="*(100))
print(f"{'Input:':10}|{len(input):^20}{' ':10}{translate_to_string(input.tolist()):50}")
print(f"{'Target:':10}|{len(target):^20}{' ':10}{translate_to_string(target.tolist()):50}")

print('\n'*3)

print("Short Sequence Dataset:")


input, target = short_seq_dataset[0]

print(f"{' ':11}{'Length':^20}{' ':10}{'Raw':^50}")
print("="*(100))
print(f"{'Input:':10}|{len(input):^20}{' ':10}{translate_to_string(input.tolist()):50}")
print(f"{'Target:':10}|{len(target):^20}{' ':10}{translate_to_string(target.tolist()):50}")

Normal Dataset:
                  Length                                        Raw                        
Input:    |         50                   first_citizen:\before_we_proceed_any_further,_hear
Target:   |         50                   irst_citizen:\before_we_proceed_any_further,_hear_




Short Sequence Dataset:
                  Length                                        Raw                        
Input:    |         25                   first_citizen:\before_we_                         
Target:   |         25                   irst_citizen:\before_we_p                         


In [16]:
print("Normal Dataloader:")


input, target = next(iter(training_dataloader))

print(f"{' ':15}{'Batch Size':>10}{' ':10}{'Sequence Length':>20}")
print("="*(60))
print(f"{'Input:':10}|{input.shape[0]:>10}{' ':20}{input.shape[1]:>10}")
print(f"{'Target:':10}|{target.shape[0]:>10}{' ':20}{target.shape[1]:>10}")


print('\n'*3)

print("Short Sequence Dataloader:")


input, target = next(iter(short_dataloader))

print(f"{' ':15}{'Batch Size':>10}{' ':10}{'Sequence Length':>20}")
print("="*(60))
print(f"{'Input:':10}|{input.shape[0]:>10}{' ':20}{input.shape[1]:>10}")
print(f"{'Target:':10}|{target.shape[0]:>10}{' ':20}{target.shape[1]:>10}")


Normal Dataloader:
               Batch Size               Sequence Length
Input:    |       256                            50
Target:   |       256                            50




Short Sequence Dataloader:
               Batch Size               Sequence Length
Input:    |       256                            25
Target:   |       256                            25
